In [ ]:
"""Deezer agent — built from deepagents-python-quickstart shape.

Quickstart source: https://docs.langchain.com/oss/python/deepagents/quickstart
Adaptation: provider = Groq (GROQ_API_KEY in .env), tools = Deezer public API.
"""

import os

from dotenv import load_dotenv

load_dotenv()

# LangSmith tracing config (traces -> projet deezer_agent)
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "deezer_agent")
os.environ.setdefault("LANGSMITH_TRACING", "true")

import requests
from deepagents import create_deep_agent
from langchain_groq import ChatGroq

# --- Tools Deezer (API publique, sans clé) ---


def search_song(query: str) -> list:
    """Cherche un titre/musique sur Deezer. Ex: 'Daft Punk Get Lucky'."""
    r = requests.get("https://api.deezer.com/search", params={"q": query}, timeout=10)
    r.raise_for_status()
    out = []
    for t in r.json().get("data", [])[:5]:
        out.append(
            {
                "titre": t["title"],
                "artiste": t["artist"]["name"],
                "album": t["album"]["title"],
                "duree_sec": t["duration"],
                "preview_mp3": t["preview"],
                "lien": t["link"],
                "id": t["id"],
            }
        )
    return out


def search_artist(nom: str) -> list:
    """Cherche un artiste sur Deezer. Ex: 'Daft Punk'."""
    r = requests.get("https://api.deezer.com/search/artist", params={"q": nom}, timeout=10)
    r.raise_for_status()
    out = []
    for a in r.json().get("data", [])[:5]:
        out.append(
            {
                "nom": a["name"],
                "id": a["id"],
                "fans": a["nb_fan"],
                "nb_albums": a["nb_album"],
                "lien": a["link"],
            }
        )
    return out


def top_titres_artist(nom_ou_id: str) -> list:
    """Top 10 titres d'un artiste Deezer. Input: nom ('Daft Punk') ou id ('27')."""
    artist_id = nom_ou_id
    if not str(nom_ou_id).isdigit():
        s = requests.get(
            "https://api.deezer.com/search/artist", params={"q": nom_ou_id}, timeout=10
        ).json()
        if not s.get("data"):
            return f"Artiste '{nom_ou_id}' introuvable."
        artist_id = s["data"][0]["id"]
    r = requests.get(f"https://api.deezer.com/artist/{artist_id}/top", params={"limit": 10}, timeout=10)
    r.raise_for_status()
    out = []
    for i, t in enumerate(r.json().get("data", []), 1):
        out.append(
            {
                "rang": i,
                "titre": t["title"],
                "artiste": t["artist"]["name"],
                "album": t["album"]["title"],
                "preview_mp3": t["preview"],
                "lien": t["link"],
            }
        )
    return out


# System prompt facon quickstart "research agent", adapte musique
deezer_instructions = """You are an expert music researcher. Your job is to conduct thorough research on artists and songs using Deezer, then write a polished report.

You have access to Deezer search tools as your primary means of gathering information.

## `search_song`
Use this to run a song search for a given query.

## `search_artist`
Use this to run an artist search for a given name.

## `top_titres_artist`
Use this to get the top tracks of an artist (name or id).
"""

model = ChatGroq(model="openai/gpt-oss-120b")

# Objet expose pour langgraph.json / Studio
agent = create_deep_agent(
    model=model,
    tools=[search_song, search_artist, top_titres_artist],
    system_prompt=deezer_instructions,
)


conversation: list = []


def send_message(text: str) -> str:
    """Envoie un message à l'agent et retourne sa réponse. Garde l'historique."""
    conversation.append({"role": "user", "content": text})
    result = agent.invoke({"messages": conversation})
    reply = result["messages"][-1].content
    conversation.append({"role": "assistant", "content": reply})
    return reply


if __name__ == "__main__":
    print("deezer_agent — discute (tape 'quit' pour sortir)")
    while True:
        try:
            text = input("toi> ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if text.lower() in ("quit", "exit", "q"):
            break
        if not text:
            continue
        print("agent>", send_message(text))
